# Geomagnetism (Chapman & Bartels, 1940): Implementation / 구현

이 노트북은 Chapman & Bartels의 "Geomagnetism"에서 제시된 핵심 수학적 방법론을 직접 구현합니다.

This notebook implements the key mathematical methodologies from Chapman & Bartels' "Geomagnetism."

**구현 내용 / Contents:**
1. 쌍극자 자기장과 자기 요소 / Dipole Field and Magnetic Elements
2. 구면 조화 분석 — Gauss 계수 / Spherical Harmonic Analysis — Gauss Coefficients
3. 내부/외부 기원 분리 / Internal vs. External Source Separation
4. Non-dipole 자기장 시각화 / Non-Dipole Field Visualization
5. K 지수 시뮬레이션 / K Index Simulation
6. Kp 지수 산출 / Kp Index Calculation
7. Dst 지수와 자기 폭풍 분해 / Dst Index and Storm Decomposition
8. 27일 재현 — Bartels Rotation Chart / 27-Day Recurrence — Bartels Rotation Chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import lpmv  # Associated Legendre functions
from matplotlib.colors import TwoSlopeNorm

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: 쌍극자 자기장과 자기 요소 / Dipole Field and Magnetic Elements

지구 자기장의 가장 기본적인 근사인 **중심 쌍극자(centered dipole)** 모델을 구현합니다.
쌍극자 포텐셜로부터 7가지 자기 요소($F$, $H$, $Z$, $X$, $Y$, $D$, $I$)를 계산하고 시각화합니다.

$$B_r = -\frac{2M\cos\theta}{r^3}, \quad B_\theta = -\frac{M\sin\theta}{r^3}$$

$$\tan I = 2\cot\theta = 2\tan\lambda_m$$

In [ ]:
def dipole_field(
    theta: np.ndarray,
    r: np.ndarray,
    M: float = 8.0e22,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute magnetic dipole field components.

    Args:
        theta: Colatitude in radians.
        r: Radial distance in meters.
        M: Dipole moment in A·m² (default: Earth's ~8.0e22).

    Returns:
        Tuple of (B_r, B_theta) in Tesla.
    """
    mu0_over_4pi = 1e-7  # T·m/A
    B_r = -2 * mu0_over_4pi * M * np.cos(theta) / r**3
    B_theta = -mu0_over_4pi * M * np.sin(theta) / r**3
    return B_r, B_theta


def magnetic_elements(
    theta: np.ndarray,
    r: float = 6.371e6,
    M: float = 8.0e22,
) -> dict[str, np.ndarray]:
    """Compute all 7 magnetic elements from the dipole model.

    Args:
        theta: Colatitude in radians.
        r: Radial distance (default: Earth's surface).
        M: Dipole moment.

    Returns:
        Dictionary with keys F, H, Z, I (inclination), and lambda_m.
    """
    B_r, B_theta = dipole_field(theta, r, M)
    # In geomagnetic coordinates: Z = -B_r (positive downward), H = -B_theta
    Z = -B_r
    H = -B_theta
    F = np.sqrt(H**2 + Z**2)
    I = np.arctan2(Z, H)  # Inclination
    lambda_m = np.pi / 2 - theta  # Geomagnetic latitude
    return {"F": F, "H": H, "Z": Z, "I": I, "lambda_m": lambda_m}


# Compute elements as a function of geomagnetic latitude
theta_arr = np.linspace(0.01, np.pi - 0.01, 500)
elems = magnetic_elements(theta_arr)
lat_deg = np.degrees(elems["lambda_m"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) H and Z components
ax = axes[0, 0]
ax.plot(lat_deg, elems["H"] * 1e9, "b-", label="H (horizontal)", linewidth=2)
ax.plot(lat_deg, elems["Z"] * 1e9, "r-", label="Z (vertical)", linewidth=2)
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Geomagnetic Latitude (°)")
ax.set_ylabel("Field Intensity (nT)")
ax.set_title("(a) Horizontal & Vertical Components / 수평·수직 성분")
ax.legend()

# (b) Total intensity F
ax = axes[0, 1]
ax.plot(lat_deg, elems["F"] * 1e9, "k-", linewidth=2)
ax.set_xlabel("Geomagnetic Latitude (°)")
ax.set_ylabel("Total Intensity F (nT)")
ax.set_title("(b) Total Field Intensity / 전자기장 강도")

# (c) Inclination I
ax = axes[1, 0]
ax.plot(lat_deg, np.degrees(elems["I"]), "g-", linewidth=2)
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Geomagnetic Latitude (°)")
ax.set_ylabel("Inclination I (°)")
ax.set_title("(c) Inclination / 복각")
ax.set_ylim(-95, 95)

# (d) Verify tan(I) = 2*tan(lambda_m) relation
ax = axes[1, 1]
I_predicted = np.arctan(2 * np.tan(elems["lambda_m"]))
ax.plot(lat_deg, np.degrees(elems["I"]), "g-", label="Direct calculation", linewidth=2)
ax.plot(lat_deg, np.degrees(I_predicted), "r--", label=r"$\tan I = 2\tan\lambda_m$", linewidth=2)
ax.set_xlabel("Geomagnetic Latitude (°)")
ax.set_ylabel("Inclination I (°)")
ax.set_title(r"(d) Verify $\tan I = 2\tan\lambda_m$ / 관계식 검증")
ax.legend()

fig.suptitle("Part 1: Dipole Magnetic Elements / 쌍극자 자기 요소", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Equatorial field H(0°) = {elems['H'][250]*1e9:.0f} nT")
print(f"Polar field Z(90°)    = {elems['Z'][0]*1e9:.0f} nT")
print(f"Ratio Z_pole/H_eq     = {abs(elems['Z'][0]/elems['H'][250]):.2f} (theory: 2.0)")

## Part 2: 구면 조화 분석 — Gauss 계수 / Spherical Harmonic Analysis — Gauss Coefficients

Gauss가 1839년에 도입하고 Chapman & Bartels가 체계화한 구면 조화 전개를 직접 구현합니다.
실제 IGRF Gauss 계수를 사용하여 지구 자기장을 재구성합니다.

$$V = a \sum_{n=1}^{N} \sum_{m=0}^{n} \left(\frac{a}{r}\right)^{n+1} \left(g_n^m \cos m\phi + h_n^m \sin m\phi\right) P_n^m(\cos\theta)$$

In [ ]:
# IGRF-13 coefficients for epoch 2020 (n=1 to 4, in nT)
# Schmidt semi-normalized convention
IGRF_COEFFS = {
    # (n, m): (g_n^m, h_n^m)
    (1, 0): (-29404.8, 0.0),
    (1, 1): (-1450.9, 4652.5),
    (2, 0): (-2499.6, 0.0),
    (2, 1): (2982.0, -2991.6),
    (2, 2): (1677.0, -734.6),
    (3, 0): (1363.2, 0.0),
    (3, 1): (-2381.2, -82.1),
    (3, 2): (1236.2, 241.9),
    (3, 3): (525.7, -543.4),
    (4, 0): (903.0, 0.0),
    (4, 1): (809.5, 281.9),
    (4, 2): (86.3, -158.4),
    (4, 3): (-309.4, 199.7),
    (4, 4): (48.0, -349.7),
}


def schmidt_legendre(n: int, m: int, theta: np.ndarray) -> np.ndarray:
    """Compute Schmidt semi-normalized associated Legendre function.

    Args:
        n: Degree.
        m: Order.
        theta: Colatitude in radians.

    Returns:
        P_n^m(cos(theta)) in Schmidt normalization.
    """
    cos_theta = np.cos(theta)
    # scipy lpmv uses unnormalized convention; convert to Schmidt
    P_raw = lpmv(m, n, cos_theta)
    if m == 0:
        return P_raw
    # Schmidt factor: S_n^m = sqrt(2 * (n-m)! / (n+m)!)
    factor = 1.0
    for k in range(n - m + 1, n + m + 1):
        factor *= k
    schmidt = np.sqrt(2.0 / factor)
    return schmidt * P_raw


def sh_magnetic_field(
    theta: np.ndarray,
    phi: np.ndarray,
    r: float = 6.371e6,
    a: float = 6.371e6,
    n_max: int = 4,
    coeffs: dict = IGRF_COEFFS,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute magnetic field from spherical harmonic expansion.

    Args:
        theta: 2D colatitude grid in radians.
        phi: 2D longitude grid in radians.
        r: Radial distance (meters).
        a: Reference radius (Earth's radius).
        n_max: Maximum degree.
        coeffs: Gauss coefficients dict.

    Returns:
        Tuple of (B_r, B_theta, B_phi) in nT.
    """
    B_r = np.zeros_like(theta)
    B_theta = np.zeros_like(theta)
    B_phi = np.zeros_like(theta)

    for n in range(1, n_max + 1):
        for m in range(0, n + 1):
            g, h = coeffs.get((n, m), (0.0, 0.0))
            P = schmidt_legendre(n, m, theta)

            # Numerical derivative of P w.r.t. theta
            dtheta = 1e-6
            P_plus = schmidt_legendre(n, m, theta + dtheta)
            P_minus = schmidt_legendre(n, m, theta - dtheta)
            dP = (P_plus - P_minus) / (2 * dtheta)

            ratio = (a / r) ** (n + 2)
            cos_m = np.cos(m * phi)
            sin_m = np.sin(m * phi)

            gh_cos_sin = g * cos_m + h * sin_m

            # B_r = -dV/dr
            B_r += (n + 1) * ratio * gh_cos_sin * P
            # B_theta = -(1/r) dV/dtheta
            B_theta += -ratio * gh_cos_sin * dP
            # B_phi = -(1/(r*sin(theta))) dV/dphi
            sin_theta = np.sin(theta)
            sin_theta_safe = np.where(np.abs(sin_theta) < 1e-10, 1e-10, sin_theta)
            B_phi += ratio * m * (-g * sin_m + h * cos_m) * P / sin_theta_safe

    return B_r, B_theta, B_phi


# Create a lat-lon grid
n_pts = 180
lat_grid = np.linspace(-89, 89, n_pts)
lon_grid = np.linspace(-180, 180, 2 * n_pts)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
THETA = np.radians(90 - LAT)  # Colatitude
PHI = np.radians(LON)

Br, Bth, Bph = sh_magnetic_field(THETA, PHI)
F_total = np.sqrt(Br**2 + Bth**2 + Bph**2)
H_horiz = np.sqrt(Bth**2 + Bph**2)
I_incl = np.degrees(np.arctan2(-Br, H_horiz))  # Z = -Br for downward positive

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# (a) Total intensity
ax = axes[0, 0]
cs = ax.contourf(LON, LAT, F_total, levels=20, cmap="viridis")
plt.colorbar(cs, ax=ax, label="nT")
ax.set_title("(a) Total Intensity F / 전자기장 강도")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# (b) Vertical component Z = -B_r
ax = axes[0, 1]
Z_comp = -Br
norm = TwoSlopeNorm(vmin=Z_comp.min(), vcenter=0, vmax=Z_comp.max())
cs = ax.contourf(LON, LAT, Z_comp, levels=20, cmap="RdBu_r", norm=norm)
plt.colorbar(cs, ax=ax, label="nT")
ax.contour(LON, LAT, Z_comp, levels=[0], colors="k", linewidths=2)
ax.set_title("(b) Vertical Component Z / 수직 성분 (dip equator: black)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# (c) Inclination I
ax = axes[1, 0]
norm_i = TwoSlopeNorm(vmin=-90, vcenter=0, vmax=90)
cs = ax.contourf(LON, LAT, I_incl, levels=np.arange(-90, 91, 10), cmap="RdBu_r", norm=norm_i)
plt.colorbar(cs, ax=ax, label="degrees")
ax.contour(LON, LAT, I_incl, levels=[0], colors="k", linewidths=2)
ax.set_title("(c) Inclination I / 복각 (magnetic equator: black)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# (d) Horizontal intensity H
ax = axes[1, 1]
cs = ax.contourf(LON, LAT, H_horiz, levels=20, cmap="inferno")
plt.colorbar(cs, ax=ax, label="nT")
ax.set_title("(d) Horizontal Intensity H / 수평 성분")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

fig.suptitle("Part 2: Spherical Harmonic Field (IGRF, n≤4) / 구면 조화 자기장", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Part 3: 내부/외부 기원 분리 & 에너지 스펙트럼 / Internal/External Separation & Power Spectrum

Gauss의 핵심 통찰: 내부 항은 $(a/r)^{n+1}$, 외부 항은 $(r/a)^n$으로 감소/증가하므로, $B_r$ 측정에서 내부 항에는 $(n+1)$, 외부 항에는 $-n$이 곱해집니다. 이 차이로 분리가 가능합니다.

또한 각 차수 $n$의 자기장 에너지를 나타내는 **Lowes-Mauersberger spectrum**을 계산합니다:

$$R_n = (n+1) \sum_{m=0}^{n} \left[(g_n^m)^2 + (h_n^m)^2\right]$$

In [ ]:
def lowes_mauersberger_spectrum(
    coeffs: dict,
    n_max: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute the Lowes-Mauersberger spatial power spectrum.

    Args:
        coeffs: Gauss coefficients dict {(n,m): (g, h)}.
        n_max: Maximum degree.

    Returns:
        Tuple of (degrees, R_n values in nT²).
    """
    degrees = np.arange(1, n_max + 1)
    R_n = np.zeros(n_max)
    for n in range(1, n_max + 1):
        power = 0.0
        for m in range(0, n + 1):
            g, h = coeffs.get((n, m), (0.0, 0.0))
            power += g**2 + h**2
        R_n[n - 1] = (n + 1) * power
    return degrees, R_n


# Extended IGRF coefficients up to n=13 for realistic spectrum
IGRF_EXTENDED = {
    (1, 0): (-29404.8, 0.0), (1, 1): (-1450.9, 4652.5),
    (2, 0): (-2499.6, 0.0), (2, 1): (2982.0, -2991.6), (2, 2): (1677.0, -734.6),
    (3, 0): (1363.2, 0.0), (3, 1): (-2381.2, -82.1), (3, 2): (1236.2, 241.9), (3, 3): (525.7, -543.4),
    (4, 0): (903.0, 0.0), (4, 1): (809.5, 281.9), (4, 2): (86.3, -158.4), (4, 3): (-309.4, 199.7), (4, 4): (48.0, -349.7),
    (5, 0): (-234.3, 0.0), (5, 1): (363.2, 47.7), (5, 2): (187.8, 208.3), (5, 3): (-140.7, -121.2), (5, 4): (-151.2, 32.3), (5, 5): (13.5, 98.9),
    (6, 0): (65.8, 0.0), (6, 1): (65.5, -19.1), (6, 2): (73.0, 25.1), (6, 3): (-121.5, 52.8), (6, 4): (-36.2, -64.5), (6, 5): (13.5, 8.9), (6, 6): (-64.7, 68.1),
    (7, 0): (80.6, 0.0), (7, 1): (-76.7, -51.5), (7, 2): (-8.2, -16.9), (7, 3): (56.5, 2.2), (7, 4): (15.8, 23.5), (7, 5): (6.4, -2.2), (7, 6): (-7.2, -27.2), (7, 7): (9.8, -1.8),
    (8, 0): (23.7, 0.0), (8, 1): (9.7, 8.4), (8, 2): (-17.6, -15.3), (8, 3): (-0.5, 12.8), (8, 4): (-21.1, -11.7), (8, 5): (15.3, 14.9), (8, 6): (13.7, 3.6), (8, 7): (-16.5, -6.9), (8, 8): (-0.3, 2.8),
    (9, 0): (5.0, 0.0), (9, 1): (8.2, -23.3), (9, 2): (2.9, 11.0), (9, 3): (-1.4, 9.8), (9, 4): (-1.1, -5.1), (9, 5): (-13.3, -6.3), (9, 6): (1.1, 7.8), (9, 7): (8.8, 0.4), (9, 8): (-9.3, -4.1), (9, 9): (-10.5, 8.6),
    (10, 0): (-1.9, 0.0), (10, 1): (-6.3, 3.2), (10, 2): (0.1, -0.6), (10, 3): (0.5, 4.6), (10, 4): (-0.5, 4.4), (10, 5): (1.8, -7.9), (10, 6): (-0.7, -0.6), (10, 7): (2.1, -4.2), (10, 8): (2.4, -2.8), (10, 9): (-1.8, -1.2), (10, 10): (-3.6, -8.7),
    (11, 0): (3.1, 0.0), (11, 1): (-1.5, -0.1), (11, 2): (-2.3, 2.0), (11, 3): (2.1, -0.7), (11, 4): (-0.9, -1.1), (11, 5): (0.6, 0.7), (11, 6): (-0.7, -0.2), (11, 7): (0.2, -2.1), (11, 8): (1.7, -1.5), (11, 9): (-0.2, -2.5), (11, 10): (0.4, -2.0), (11, 11): (3.5, -2.3),
    (12, 0): (-2.0, 0.0), (12, 1): (-0.3, -1.0), (12, 2): (0.4, 0.5), (12, 3): (1.3, 1.8), (12, 4): (-0.9, -2.2), (12, 5): (0.9, 0.3), (12, 6): (0.1, 0.7), (12, 7): (0.5, -0.1), (12, 8): (0.6, 0.2), (12, 9): (-0.9, 0.3), (12, 10): (0.3, -0.2), (12, 11): (-0.2, -0.5), (12, 12): (-0.3, 0.6),
    (13, 0): (0.1, 0.0), (13, 1): (-0.9, -0.2), (13, 2): (0.6, -0.5), (13, 3): (0.7, 0.5), (13, 4): (-0.3, -0.8), (13, 5): (0.3, 0.5), (13, 6): (-0.4, 0.1), (13, 7): (0.2, 0.0), (13, 8): (-0.4, 0.4), (13, 9): (-0.3, -0.1), (13, 10): (0.0, 0.3), (13, 11): (-0.1, -0.4), (13, 12): (0.3, -0.1), (13, 13): (-0.1, -0.3),
}

degrees, R_n = lowes_mauersberger_spectrum(IGRF_EXTENDED, n_max=13)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Power spectrum (log scale)
ax = axes[0]
ax.semilogy(degrees, R_n, "ko-", markersize=8, linewidth=2)
ax.set_xlabel("Degree n")
ax.set_ylabel(r"$R_n$ (nT²)")
ax.set_title("(a) Lowes-Mauersberger Spectrum / 에너지 스펙트럼")
ax.set_xticks(degrees)

# Annotate dipole dominance
pct_dipole = R_n[0] / R_n.sum() * 100
ax.annotate(
    f"n=1 (dipole)\n{pct_dipole:.1f}% of total",
    xy=(1, R_n[0]), xytext=(4, R_n[0]),
    arrowprops=dict(arrowstyle="->"), fontsize=10
)

# (b) Cumulative fraction
ax = axes[1]
cumulative = np.cumsum(R_n) / np.sum(R_n) * 100
ax.bar(degrees, cumulative, color="steelblue", alpha=0.7, edgecolor="k")
ax.axhline(99, color="r", linestyle="--", label="99%")
ax.set_xlabel("Degree n (cumulative up to)")
ax.set_ylabel("Cumulative Power (%)")
ax.set_title("(b) Cumulative Power / 누적 에너지")
ax.set_xticks(degrees)
ax.legend()

fig.suptitle("Part 3: Gauss Coefficient Power Spectrum / Gauss 계수 에너지 스펙트럼", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Dipole (n=1) power fraction: {pct_dipole:.2f}%")
print(f"n≤3 captures: {np.sum(R_n[:3])/np.sum(R_n)*100:.2f}%")
print(f"n≤4 captures: {np.sum(R_n[:4])/np.sum(R_n)*100:.2f}%")

## Part 4: Non-Dipole 자기장 시각화 / Non-Dipole Field Visualization

전체 자기장에서 쌍극자($n=1$) 성분을 제거하면 **non-dipole field**가 남습니다.
이 잔여장은 남대서양 이상(South Atlantic Anomaly) 등 외핵의 국소적 대류 패턴을 반영합니다.

In [ ]:
# Full field (n=1 to 4) vs dipole-only (n=1)
Br_full, Bth_full, Bph_full = sh_magnetic_field(THETA, PHI, n_max=4)
Br_dip, Bth_dip, Bph_dip = sh_magnetic_field(THETA, PHI, n_max=1)

# Non-dipole = full - dipole
Br_nd = Br_full - Br_dip
Bth_nd = Bth_full - Bth_dip
Bph_nd = Bph_full - Bph_dip

F_full = np.sqrt(Br_full**2 + Bth_full**2 + Bph_full**2)
F_dip = np.sqrt(Br_dip**2 + Bth_dip**2 + Bph_dip**2)
F_nd = F_full - F_dip  # Residual total intensity

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Full field F
ax = axes[0]
cs = ax.contourf(LON, LAT, F_full, levels=20, cmap="viridis")
plt.colorbar(cs, ax=ax, label="nT")
ax.set_title("(a) Full Field F (n≤4)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# (b) Dipole-only F
ax = axes[1]
cs = ax.contourf(LON, LAT, F_dip, levels=20, cmap="viridis")
plt.colorbar(cs, ax=ax, label="nT")
ax.set_title("(b) Dipole Only (n=1)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# (c) Non-dipole residual
ax = axes[2]
vmax = max(abs(F_nd.min()), abs(F_nd.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cs = ax.contourf(LON, LAT, F_nd, levels=20, cmap="RdBu_r", norm=norm)
plt.colorbar(cs, ax=ax, label="nT")
ax.set_title("(c) Non-Dipole Residual (n=2–4)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")

# Mark South Atlantic Anomaly region
ax.plot(-50, -30, "k*", markersize=15)
ax.annotate("SAA", xy=(-50, -30), xytext=(-20, -50), fontsize=10,
            arrowprops=dict(arrowstyle="->"))

fig.suptitle("Part 4: Non-Dipole Field / 비쌍극자장", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Non-dipole field range: {F_nd.min():.0f} to {F_nd.max():.0f} nT")
print(f"Non-dipole / Full field: {np.std(F_nd)/np.mean(F_full)*100:.1f}% (RMS)")

## Part 5: K 지수 시뮬레이션 / K Index Simulation

Bartels의 K 지수 산출 과정을 시뮬레이션합니다:
1. 합성 magnetogram(자기 기록) 생성 — Sq 변동 + 랜덤 교란
2. 3시간 구간의 range 측정
3. 준대수 척도(quasi-logarithmic scale)로 변환

| $K$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|-----|---|---|---|---|---|---|---|---|---|---|
| $a_K$ (nT) | 0–5 | 5–10 | 10–20 | 20–40 | 40–70 | 70–120 | 120–200 | 200–330 | 330–500 | >500 |

In [ ]:
# K-index thresholds for a mid-latitude station (nT)
K_THRESHOLDS = [5, 10, 20, 40, 70, 120, 200, 330, 500]


def amplitude_to_K(amplitude: float, thresholds: list = K_THRESHOLDS) -> int:
    """Convert disturbance amplitude to K index.

    Args:
        amplitude: Peak-to-peak range in nT.
        thresholds: K-index boundary values.

    Returns:
        K value (0–9).
    """
    for k, threshold in enumerate(thresholds):
        if amplitude < threshold:
            return k
    return 9


def generate_magnetogram(
    hours: int = 72,
    sq_amplitude: float = 30.0,
    storm_peak: float = -150.0,
    storm_onset: float = 24.0,
    noise_level: float = 5.0,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Generate a synthetic magnetogram with Sq variation and storm.

    Args:
        hours: Duration in hours.
        sq_amplitude: Sq daily variation amplitude (nT).
        storm_peak: Storm main phase minimum (nT).
        storm_onset: Storm onset time (hours).
        noise_level: Random noise level (nT).
        seed: Random seed.

    Returns:
        Tuple of (time, total_signal, sq_component, disturbance_component).
    """
    rng = np.random.default_rng(seed)
    dt = 1 / 60  # 1-minute resolution
    t = np.arange(0, hours, dt)

    # Sq variation (diurnal + semidiurnal)
    sq = sq_amplitude * (
        0.7 * np.sin(2 * np.pi * t / 24 - np.pi / 3)
        + 0.3 * np.sin(4 * np.pi * t / 24 + np.pi / 6)
    )

    # Magnetic storm: SC + main phase + recovery
    storm = np.zeros_like(t)
    t_rel = t - storm_onset
    # SC: sharp positive pulse
    sc_mask = (t_rel >= 0) & (t_rel < 0.5)
    storm[sc_mask] = 40 * np.exp(-((t_rel[sc_mask] - 0.15) ** 2) / 0.02)
    # Main phase: exponential decrease
    main_mask = t_rel >= 0.5
    storm[main_mask] = storm_peak * (1 - np.exp(-(t_rel[main_mask] - 0.5) / 3))
    # Recovery: exponential return
    recovery_start = 8.0
    rec_mask = t_rel >= recovery_start
    storm_at_rec = storm_peak * (1 - np.exp(-(recovery_start - 0.5) / 3))
    storm[rec_mask] = storm_at_rec * np.exp(-(t_rel[rec_mask] - recovery_start) / 12)

    # Random noise
    noise = noise_level * rng.standard_normal(len(t))

    disturbance = storm + noise
    total = sq + disturbance

    return t, total, sq, disturbance


# Generate magnetogram
t_hrs, total, sq, disturbance = generate_magnetogram()

# Calculate K indices for each 3-hour interval
n_intervals = int(t_hrs[-1] // 3)
K_values = []
K_times = []
for i in range(n_intervals):
    t_start = i * 3
    t_end = (i + 1) * 3
    mask = (t_hrs >= t_start) & (t_hrs < t_end)
    # Remove Sq to get disturbance range
    dist_segment = disturbance[mask]
    amplitude = dist_segment.max() - dist_segment.min()
    K_values.append(amplitude_to_K(amplitude))
    K_times.append(t_start + 1.5)  # Midpoint

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# (a) Raw magnetogram
ax = axes[0]
ax.plot(t_hrs, total, "b-", linewidth=0.5, alpha=0.7, label="Total (Sq + disturbance)")
ax.plot(t_hrs, sq, "g--", linewidth=1.5, label="Sq variation")
ax.set_ylabel("ΔH (nT)")
ax.set_title("(a) Synthetic Magnetogram / 합성 자기기록")
ax.legend()
ax.axvline(24, color="r", linestyle=":", alpha=0.5, label="Storm onset")

# (b) Disturbance only (Sq removed)
ax = axes[1]
ax.plot(t_hrs, disturbance, "r-", linewidth=0.5)
ax.set_ylabel("Disturbance (nT)")
ax.set_title("(b) Disturbance Component (Sq removed) / 교란 성분")
# Mark 3-hour intervals
for i in range(n_intervals + 1):
    ax.axvline(i * 3, color="gray", linewidth=0.3, alpha=0.5)

# (c) K indices
ax = axes[2]
colors = plt.cm.YlOrRd(np.array(K_values) / 9)
ax.bar(K_times, K_values, width=2.8, color=colors, edgecolor="k", linewidth=0.5)
ax.set_ylabel("K index")
ax.set_xlabel("Time (hours)")
ax.set_title("(c) 3-Hourly K Index / 3시간 K 지수")
ax.set_yticks(range(10))
ax.set_ylim(0, 9.5)

fig.suptitle("Part 5: K Index Simulation / K 지수 시뮬레이션", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("K values:", K_values)
print(f"Max K = {max(K_values)} (storm peak interval)")

## Part 6: Kp 지수 산출 / Kp Index Calculation

전 세계 여러 관측소의 $K$ 값을 **표준화(standardize)**하고 평균하여 $K_p$를 산출합니다.
관측소마다 다른 K 임계값을 사용하여, 위도/지역 효과를 보정하는 과정을 시뮬레이션합니다.

$$K_p = \frac{1}{N}\sum_{i=1}^{N} K_{s,i}$$

In [ ]:
# Kp to ap conversion table (Bartels)
KP_TO_AP = {
    0: 0, 0.33: 2, 0.67: 3, 1: 4, 1.33: 5, 1.67: 6, 2: 7,
    2.33: 9, 2.67: 12, 3: 15, 3.33: 18, 3.67: 22, 4: 27,
    4.33: 32, 4.67: 39, 5: 48, 5.33: 56, 5.67: 67, 6: 80,
    6.33: 94, 6.67: 111, 7: 132, 7.33: 154, 7.67: 179, 8: 207,
    8.33: 236, 8.67: 300, 9: 400,
}


def kp_to_ap(kp: float) -> float:
    """Convert Kp to ap using the standard lookup table.

    Args:
        kp: Kp value (0 to 9, in 1/3 steps).

    Returns:
        Corresponding ap value in nT.
    """
    keys = sorted(KP_TO_AP.keys())
    # Find nearest Kp value
    nearest = min(keys, key=lambda x: abs(x - kp))
    return KP_TO_AP[nearest]


def simulate_multi_station_kp(
    n_stations: int = 13,
    n_intervals: int = 24,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Simulate Kp calculation from multiple stations.

    Args:
        n_stations: Number of mid-latitude stations.
        n_intervals: Number of 3-hour intervals (3 days = 24).
        seed: Random seed.

    Returns:
        Tuple of (station_K, station_Ks, Kp, ap_values).
    """
    rng = np.random.default_rng(seed)

    # Station properties: different latitudes → different sensitivity
    lat_factors = 0.7 + 0.6 * rng.random(n_stations)  # Scale factors

    # "True" global activity (arbitrary units)
    # Quiet period → storm → recovery
    true_activity = np.zeros(n_intervals)
    true_activity[:8] = rng.exponential(10, 8)  # Quiet day
    true_activity[8:10] = [80, 150]  # Storm onset
    true_activity[10:14] = [200, 180, 120, 90]  # Main phase
    true_activity[14:18] = [60, 40, 30, 20]  # Recovery
    true_activity[18:] = rng.exponential(8, n_intervals - 18)  # Return to quiet

    # Each station measures with different sensitivity + noise
    station_amplitudes = np.outer(lat_factors, true_activity)
    station_amplitudes += rng.normal(0, 5, station_amplitudes.shape)
    station_amplitudes = np.clip(station_amplitudes, 0, None)

    # Raw K at each station
    station_K = np.zeros_like(station_amplitudes, dtype=int)
    for i in range(n_stations):
        # Each station has scaled thresholds
        scaled_thresholds = [t * lat_factors[i] for t in K_THRESHOLDS]
        for j in range(n_intervals):
            station_K[i, j] = amplitude_to_K(station_amplitudes[i, j], scaled_thresholds)

    # Standardized Ks: apply inverse of station factor
    station_Ks = np.zeros_like(station_amplitudes)
    for i in range(n_stations):
        for j in range(n_intervals):
            station_Ks[i, j] = amplitude_to_K(station_amplitudes[i, j])

    # Kp = mean of standardized K, rounded to nearest 1/3
    Kp_raw = station_Ks.mean(axis=0)
    Kp = np.round(Kp_raw * 3) / 3  # Round to nearest 1/3

    # ap values
    ap_values = np.array([kp_to_ap(k) for k in Kp])

    return station_K, station_Ks, Kp, ap_values


station_K, station_Ks, Kp, ap_vals = simulate_multi_station_kp()
time_labels = np.arange(len(Kp)) * 3 + 1.5

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# (a) Raw K from all stations (heatmap)
ax = axes[0, 0]
im = ax.imshow(station_K, aspect="auto", cmap="YlOrRd", vmin=0, vmax=9,
               extent=[0, len(Kp)*3, 13, 0])
plt.colorbar(im, ax=ax, label="K value")
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Station #")
ax.set_title("(a) Raw K at 13 Stations / 13개 관측소의 원시 K")

# (b) Standardized Ks
ax = axes[0, 1]
im = ax.imshow(station_Ks, aspect="auto", cmap="YlOrRd", vmin=0, vmax=9,
               extent=[0, len(Kp)*3, 13, 0])
plt.colorbar(im, ax=ax, label="Ks value")
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Station #")
ax.set_title("(b) Standardized Ks / 표준화된 Ks")

# (c) Kp index
ax = axes[1, 0]
colors_kp = plt.cm.YlOrRd(Kp / 9)
ax.bar(time_labels, Kp, width=2.8, color=colors_kp, edgecolor="k", linewidth=0.5)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Kp")
ax.set_title("(c) Planetary Kp Index / 전 지구 Kp 지수")
ax.set_ylim(0, 9.5)
# Add ±σ from stations
ax.errorbar(time_labels, Kp, yerr=station_Ks.std(axis=0), fmt="none",
            ecolor="gray", capsize=2, alpha=0.5)

# (d) ap index (linear scale)
ax = axes[1, 1]
ax.bar(time_labels, ap_vals, width=2.8, color="steelblue", edgecolor="k", linewidth=0.5)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("ap (nT)")
ax.set_title("(d) ap Index (linear equivalent) / ap 지수 (선형 등가)")

# Annotate Ap (daily average)
for day in range(3):
    daily_ap = ap_vals[day*8:(day+1)*8].mean()
    ax.text(day*24 + 12, max(ap_vals)*0.9, f"Ap={daily_ap:.0f}",
            ha="center", fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

fig.suptitle("Part 6: Kp Index — From Station K to Planetary Index / K에서 Kp로", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Kp range: {Kp.min():.1f} – {Kp.max():.1f}")
print(f"Daily Ap: {[ap_vals[d*8:(d+1)*8].mean():.0f} for d in range(3)]}")

## Part 7: Dst 지수와 자기 폭풍 분해 / Dst Index and Storm Decomposition

Chapman & Bartels의 자기 폭풍 분해를 구현합니다:

$$\Delta H(t, \lambda, \tau) = D_{st}(t)\cos\lambda + S_D(\tau, \lambda)$$

- $D_{st}$: 축대칭 교란 (환전류 / ring current)
- $S_D$: local time 의존 교란 (부분 환전류, 전리층 전류)
- $\cos\lambda$ 보정: 환전류 효과의 위도 의존성

In [ ]:
def synthetic_dst(
    t_hours: np.ndarray,
    sc_amplitude: float = 30.0,
    main_phase_min: float = -200.0,
    main_phase_tau: float = 4.0,
    recovery_tau: float = 15.0,
    storm_onset: float = 6.0,
) -> np.ndarray:
    """Generate a synthetic Dst profile for a magnetic storm.

    Args:
        t_hours: Time array in hours.
        sc_amplitude: Sudden commencement amplitude (nT).
        main_phase_min: Main phase minimum Dst (nT).
        main_phase_tau: Main phase time constant (hours).
        recovery_tau: Recovery time constant (hours).
        storm_onset: Storm onset time (hours).

    Returns:
        Dst values in nT.
    """
    dst = np.zeros_like(t_hours)
    t_rel = t_hours - storm_onset

    # Sudden commencement (SC)
    sc_mask = (t_rel >= 0) & (t_rel < 1.0)
    dst[sc_mask] = sc_amplitude * np.sin(np.pi * t_rel[sc_mask] / 1.0)

    # Main phase: exponential decrease
    main_start = 1.0
    main_mask = t_rel >= main_start
    dst[main_mask] = main_phase_min * (1 - np.exp(-(t_rel[main_mask] - main_start) / main_phase_tau))

    # Recovery phase
    recovery_start = main_start + 3 * main_phase_tau
    rec_mask = t_rel >= recovery_start
    dst_at_rec = main_phase_min * (1 - np.exp(-(recovery_start - main_start) / main_phase_tau))
    dst[rec_mask] = dst_at_rec * np.exp(-(t_rel[rec_mask] - recovery_start) / recovery_tau)

    return dst


def simulate_dst_stations(
    t_hours: np.ndarray,
    dst_true: np.ndarray,
    n_stations: int = 4,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate Dst calculation from equatorial stations.

    Args:
        t_hours: Time array.
        dst_true: True Dst profile.
        n_stations: Number of equatorial stations.
        seed: Random seed.

    Returns:
        Tuple of (station_dH, station_latitudes, calculated_Dst).
    """
    rng = np.random.default_rng(seed)

    # Station latitudes (low latitude, ~10-30° geomagnetic)
    latitudes = np.array([10, 15, -20, 25]) * np.pi / 180

    # Local time offsets (stations distributed in longitude)
    lt_offsets = np.array([0, 6, 12, 18])  # Hours

    station_dH = np.zeros((n_stations, len(t_hours)))
    for i in range(n_stations):
        # Dst component: cos(lambda) factor
        dst_component = dst_true * np.cos(latitudes[i])

        # SD component: local-time dependent asymmetry
        local_time = (t_hours + lt_offsets[i]) % 24
        sd_component = 20 * np.sin(2 * np.pi * local_time / 24) * (dst_true / dst_true.min())
        sd_component = np.nan_to_num(sd_component)

        # Add noise
        noise = rng.normal(0, 3, len(t_hours))

        station_dH[i] = dst_component + sd_component + noise

    # Calculate Dst: average of cos(lambda)-corrected dH
    corrected = np.zeros_like(station_dH)
    for i in range(n_stations):
        corrected[i] = station_dH[i] / np.cos(latitudes[i])

    calculated_dst = corrected.mean(axis=0)

    return station_dH, latitudes, calculated_dst


# Generate storm
t = np.linspace(0, 72, 4320)  # 1-min resolution
dst_true = synthetic_dst(t, main_phase_min=-200)
station_dH, lats, dst_calc = simulate_dst_stations(t, dst_true)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# (a) Individual station measurements
ax = axes[0]
station_names = ["Honolulu (10°)", "Kakioka (15°)", "Hermanus (-20°)", "San Juan (25°)"]
for i in range(4):
    ax.plot(t, station_dH[i], linewidth=0.8, alpha=0.7, label=station_names[i])
ax.set_ylabel("ΔH (nT)")
ax.set_title("(a) Individual Station Measurements / 개별 관측소 측정")
ax.legend(fontsize=9)

# (b) cos(lambda) corrected
ax = axes[1]
for i in range(4):
    corrected_i = station_dH[i] / np.cos(lats[i])
    ax.plot(t, corrected_i, linewidth=0.8, alpha=0.7, label=f"Corrected {station_names[i]}")
ax.set_ylabel("ΔH / cos(λ) (nT)")
ax.set_title(r"(b) $\cos\lambda$-Corrected Measurements / $\cos\lambda$ 보정 후")
ax.legend(fontsize=9)

# (c) Dst comparison
ax = axes[2]
ax.plot(t, dst_true, "k-", linewidth=2, label="True Dst")
ax.plot(t, dst_calc, "r--", linewidth=1.5, alpha=0.8, label="Calculated Dst")
ax.fill_between(t, dst_true, alpha=0.1, color="k")
ax.set_ylabel("Dst (nT)")
ax.set_xlabel("Time (hours)")
ax.set_title("(c) Dst Index: True vs Calculated / Dst 지수: 실제 vs 계산")
ax.legend()

# Annotate storm phases
ax.annotate("SC", xy=(7, 30), fontsize=11, fontweight="bold", color="green")
ax.annotate("Main Phase", xy=(15, -150), fontsize=11, fontweight="bold", color="red")
ax.annotate("Recovery", xy=(40, -80), fontsize=11, fontweight="bold", color="blue")

# Storm classification thresholds
ax.axhline(-50, color="orange", linestyle=":", alpha=0.5)
ax.axhline(-100, color="red", linestyle=":", alpha=0.5)
ax.axhline(-250, color="purple", linestyle=":", alpha=0.5)
ax.text(70, -45, "Moderate", fontsize=8, ha="right", color="orange")
ax.text(70, -95, "Intense", fontsize=8, ha="right", color="red")
ax.text(70, -245, "Super", fontsize=8, ha="right", color="purple")

fig.suptitle("Part 7: Dst Index and Storm Decomposition / Dst 지수와 폭풍 분해", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"True Dst minimum: {dst_true.min():.0f} nT")
print(f"Calculated Dst minimum: {dst_calc.min():.0f} nT")
print(f"RMS error: {np.sqrt(np.mean((dst_true - dst_calc)**2)):.1f} nT")

## Part 8: 27일 재현 — Bartels Rotation Chart / 27-Day Recurrence — Bartels Rotation Chart

Bartels가 발명한 **회전 차트(rotation chart)**: 자기 활동 데이터를 **27일(태양 자전 주기)** 간격으로 줄 바꿈하여 표시하면, 재현 패턴이 **수직 줄무늬**로 나타납니다.

코로나 홀(coronal hole)에서 기원하는 고속 태양풍이 태양 자전에 따라 반복적으로 지구를 스치며 지나가는 효과를 시각화합니다.

In [ ]:
def generate_recurring_activity(
    n_days: int = 270,
    solar_rotation: float = 27.0,
    n_active_regions: int = 2,
    seed: int = 42,
) -> np.ndarray:
    """Generate synthetic daily Ap with 27-day recurrence.

    Args:
        n_days: Number of days to simulate.
        solar_rotation: Solar rotation period in days.
        n_active_regions: Number of persistent active regions/coronal holes.
        seed: Random seed.

    Returns:
        Daily Ap values.
    """
    rng = np.random.default_rng(seed)
    days = np.arange(n_days)

    # Base quiet-day activity
    ap = rng.exponential(5, n_days)

    # Add recurring activity from persistent coronal holes / active regions
    for region in range(n_active_regions):
        # Each region has a phase and width
        phase = rng.uniform(0, solar_rotation)
        width = rng.uniform(3, 7)  # Days of enhanced activity

        for rotation in range(n_days // int(solar_rotation) + 1):
            center = phase + rotation * solar_rotation
            # Gaussian enhancement centered on this region's passage
            enhancement = 60 * rng.uniform(0.5, 1.5) * np.exp(
                -((days - center) ** 2) / (2 * width**2)
            )
            ap += enhancement

    # Add occasional random storms (not recurring)
    n_storms = rng.poisson(3)
    for _ in range(n_storms):
        storm_day = rng.integers(0, n_days)
        storm_ap = rng.uniform(80, 200)
        ap[storm_day] = max(ap[storm_day], storm_ap)
        # Recovery over next few days
        for d in range(1, 4):
            if storm_day + d < n_days:
                ap[storm_day + d] = max(ap[storm_day + d], storm_ap * np.exp(-d / 2))

    return np.clip(ap, 0, 400)


# Generate synthetic data
n_days = 270  # 10 solar rotations
ap_daily = generate_recurring_activity(n_days)

# Create Bartels rotation chart
rotation_period = 27
n_rotations = n_days // rotation_period
chart_data = ap_daily[:n_rotations * rotation_period].reshape(n_rotations, rotation_period)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# (a) Bartels Rotation Chart
ax = axes[0]
im = ax.imshow(
    chart_data, aspect="auto", cmap="YlOrRd",
    extent=[1, 27, n_rotations, 0],
    vmin=0, vmax=100,
)
plt.colorbar(im, ax=ax, label="Ap (nT)")
ax.set_xlabel("Day within Rotation")
ax.set_ylabel("Rotation Number")
ax.set_title("(a) Bartels Rotation Chart / Bartels 회전 차트")

# Draw vertical lines where recurring activity appears
# These should appear as vertical streaks in the chart
ax.set_xlim(1, 27)

# (b) Time series with 27-day markers
ax = axes[1]
days = np.arange(n_days)
ax.plot(days, ap_daily, "k-", linewidth=0.5, alpha=0.5)
ax.fill_between(days, 0, ap_daily, alpha=0.3, color="red")
ax.set_xlabel("Day")
ax.set_ylabel("Ap (nT)")
ax.set_title("(b) Daily Ap Time Series / 일일 Ap 시계열")

# Mark 27-day intervals
for i in range(1, n_rotations + 1):
    ax.axvline(i * 27, color="blue", linestyle=":", alpha=0.3)

ax.set_xlim(0, n_days)

fig.suptitle("Part 8: 27-Day Recurrence — Bartels Rotation Chart / 27일 재현", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Autocorrelation to detect 27-day period
from numpy.fft import fft, fftfreq

# Compute autocorrelation
ap_centered = ap_daily - ap_daily.mean()
acf = np.correlate(ap_centered, ap_centered, mode="full")
acf = acf[len(acf)//2:]  # Keep positive lags
acf /= acf[0]  # Normalize

fig, ax = plt.subplots(figsize=(12, 4))
lags = np.arange(len(acf))
ax.plot(lags, acf, "b-", linewidth=1)
ax.axvline(27, color="r", linestyle="--", linewidth=2, label="27-day (solar rotation)")
ax.axvline(54, color="r", linestyle="--", linewidth=1, alpha=0.5, label="54-day (2nd harmonic)")
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Autocorrelation")
ax.set_title("Autocorrelation of Ap — Detecting 27-Day Recurrence / Ap 자기상관 — 27일 재현 탐지")
ax.set_xlim(0, 100)
ax.legend()
ax.axhline(0, color="k", linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"Autocorrelation at lag=27: {acf[27]:.3f}")
print(f"Autocorrelation at lag=54: {acf[54]:.3f}")
print(f"Peak autocorrelation lag (1-100): {np.argmax(acf[1:100])+1} days")

## Summary / 요약

| Part | 구현 내용 / Implementation | Chapman & Bartels의 기여 / Contribution |
|------|--------------------------|----------------------------------------|
| 1 | 쌍극자 자기장과 7가지 자기 요소 | 자기 요소의 체계적 정의, $\tan I = 2\tan\lambda_m$ |
| 2 | 구면 조화 전개 (IGRF 계수 사용) | Gauss 분석의 체계화, 등자기선도 |
| 3 | Lowes-Mauersberger 에너지 스펙트럼 | 내부/외부 기원 분리의 수학적 기초 |
| 4 | Non-dipole 자기장 시각화 | 쌍극자 이외 성분의 분석, 이상 영역 식별 |
| 5 | K 지수 시뮬레이션 | 3시간 교란 범위 → 준대수 척도 변환 |
| 6 | Kp 지수 산출 (다중 관측소) | 전 지구적 표준화 지수, ap/Ap 변환 |
| 7 | Dst 지수와 자기 폭풍 분해 | $D_{st}\cos\lambda + S_D$ 분해, 환전류 진단 |
| 8 | 27일 재현과 Bartels 회전 차트 | M-region 발견, 태양 자전 주기의 지자기 효과 |

**다음 논문**: Parker (1958) — 태양풍 이론. Chapman & Bartels의 27일 재현과 M-region을 **태양풍**이라는 물리적 실체로 설명합니다.